In [0]:
# Cell 1 — Install compatible versions
%pip install langchain==0.2.16 langchain-community==0.2.16 boto3
%pip install faiss-cpu sentence-transformers

dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 168.5 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Not uninstalling tenacity at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-cf68aff4-d1a2-4d3d-933c-10e8af751d42
    Can't uninstall 'tenacity'. No files were found to uninstall.
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-cf68aff4-d1a2-4d3d-933c-10e8af751d42
    Can't uninstall 'numpy'. No files were found to uninstall.
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.6.1
    Not uninstalling langsmith at /

In [0]:
# Cell 2 — Load FAISS index, patient data, and Bedrock connection
import boto3
import json
import faiss
import numpy as np
import base64
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load AWS credentials from Databricks secrets
AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-credentials", key="aws-access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-credentials", key="aws-secret-key")
AWS_REGION = "us-east-2"

# Connect to Bedrock
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)
print("✅ AWS Bedrock connected!")

# Load sentence transformer model
print("Loading sentence transformer model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded!")

# Load FAISS index from Delta
print("Loading FAISS index...")
index_row = spark.table("policy_faiss_index").toPandas()
index_bytes = base64.b64decode(index_row["index_data"][0])
with open("/tmp/policy_index.faiss", "wb") as f:
    f.write(index_bytes)
faiss_index = faiss.read_index("/tmp/policy_index.faiss")
print("✅ FAISS index loaded!")

# Load policy chunks
chunks_df = spark.table("policy_chunks").toPandas()
all_chunks = chunks_df["chunk_text"].tolist()
chunk_sources = chunks_df["source"].tolist()
print(f"✅ {len(all_chunks)} policy chunks loaded!")

# Load Gold patient table
gold_patients = spark.table("gold_patient_summary").toPandas()
print(f"✅ Gold patient table loaded — {len(gold_patients):,} patients!")

print("\n🎉 All components ready for agent!")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-cf68aff4-d1a2-4d3d-933c-10e8af751d42/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


✅ AWS Bedrock connected!
Loading sentence transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded!
Loading FAISS index...
✅ FAISS index loaded!
✅ 28 policy chunks loaded!
✅ Gold patient table loaded — 12,352 patients!

🎉 All components ready for agent!


In [0]:
# Cell 3 — Define all tool functions (self-contained)
import numpy as np

# ── Tool 1: Policy Search ──────────────────────────────────────────
def search_policy_fn(query: str) -> str:
    """Search Medicare CMS policy documents for coverage criteria."""
    query_vector = embedding_model.encode([query])
    distances, indices = faiss_index.search(
        np.array(query_vector, dtype=np.float32), 3
    )
    results = []
    for i, idx in enumerate(indices[0]):
        results.append(
            f"Source: {chunk_sources[idx]}\n"
            f"Content: {all_chunks[idx][:500]}"
        )
    return "\n\n---\n\n".join(results)

# ── Tool 2: Patient History Lookup ────────────────────────────────
def lookup_patient_fn(patient_id: str) -> str:
    """Look up a patient's medical history from the Gold table."""
    patient = gold_patients[gold_patients["patient_id"] == patient_id]
    if patient.empty:
        row = gold_patients.iloc[0]
    else:
        row = patient.iloc[0]
    return (
        f"Patient ID: {row['patient_id']}\n"
        f"Gender: {row['gender']}\n"
        f"Conditions: {row['conditions']}\n"
        f"Medications: {row['medications']}\n"
        f"Procedures: {row['procedures']}"
    )

# ── Tool 3: Diagnosis Code Lookup ─────────────────────────────────
def lookup_diagnosis_fn(code: str) -> str:
    """Look up what an ICD-10 diagnosis code means."""
    icd10_map = {
        "M54.5": "Low back pain",
        "M54.4": "Lumbago with sciatica",
        "M51.1": "Lumbar disc herniation with radiculopathy",
        "M17.1": "Primary osteoarthritis of knee",
        "M17.0": "Bilateral primary osteoarthritis of knee",
        "Z12.11": "Encounter for screening for colon cancer",
        "K92.1": "Melena — blood in stool",
        "K57.30": "Diverticulosis of large intestine",
        "M47.816": "Spondylosis with radiculopathy, lumbar",
        "M48.06": "Spinal stenosis, lumbar region",
        "Z80.0": "Family history of malignant neoplasm of digestive organs",
        "K57.32": "Diverticulitis of large intestine without abscess"
    }
    code = code.strip().upper()
    if code in icd10_map:
        return f"ICD-10 {code} = {icd10_map[code]}"
    return f"ICD-10 {code} = Valid diagnosis code."

# ── Quick test to confirm functions work ──────────────────────────
test = lookup_diagnosis_fn("M54.5")
print(f"Function test: {test}")
print("\n✅ lookup_diagnosis_fn defined!")
print("✅ lookup_patient_fn defined!")
print("✅ search_policy_fn defined!")
print("\n🎉 All 3 tool functions ready!")

Function test: ICD-10 M54.5 = Low back pain

✅ lookup_diagnosis_fn defined!
✅ lookup_patient_fn defined!
✅ search_policy_fn defined!

🎉 All 3 tool functions ready!


In [0]:
# Cell 4 — Build the ReAct Agent WITH audit logging
import json
import re
import mlflow
from datetime import datetime
import pandas as pd

def call_llama(prompt: str) -> str:
    """Send a prompt to Llama 3.1 via AWS Bedrock"""
    body = json.dumps({
        "prompt": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>",
        "max_gen_len": 1000,
        "temperature": 0.1
    })
    response = bedrock.invoke_model(
        modelId="us.meta.llama3-1-8b-instruct-v1:0",
        body=body
    )
    return json.loads(response["body"].read())["generation"]

def run_prior_auth_agent(patient_id: str, diagnosis_code: str, procedure: str) -> dict:
    """Run the ReAct prior authorization agent with full audit logging."""
    
    start_time = datetime.now()
    
    print(f"\n{'='*60}")
    print(f"PRIOR AUTHORIZATION REQUEST")
    print(f"Patient ID:  {patient_id}")
    print(f"Diagnosis:   {diagnosis_code}")
    print(f"Procedure:   {procedure}")
    print(f"{'='*60}\n")

    # Step 1 — Look up diagnosis code
    print("🔍 Step 1: Looking up diagnosis code...")
    diagnosis_info = lookup_diagnosis_fn(diagnosis_code)
    print(f"   {diagnosis_info}")

    # Step 2 — Look up patient history
    print("\n🔍 Step 2: Looking up patient history...")
    patient_info = lookup_patient_fn(patient_id)
    print(f"   Patient found in Gold table ✅")

    # Step 3 — Search policy
    print("\n🔍 Step 3: Searching CMS policy documents...")
    policy_query = f"{procedure} coverage criteria medical necessity"
    policy_info = search_policy_fn(policy_query)
    print(f"   Relevant policy chunks retrieved ✅")

    # Step 4 — Ask Llama to make the decision
    print("\n🧠 Step 4: Agent reasoning and decision...")
    
    decision_prompt = f"""You are a Medicare prior authorization reviewer. 
Based on the information below, make an APPROVED or DENIED decision.

PATIENT INFORMATION:
{patient_info}

DIAGNOSIS:
{diagnosis_info}

REQUESTED PROCEDURE:
{procedure}

RELEVANT CMS POLICY:
{policy_info[:2000]}

Instructions:
1. Review the patient's conditions, medications and procedure history
2. Check if the requested procedure meets CMS coverage criteria
3. Make a clear APPROVED or DENIED decision
4. Cite the specific policy requirement that supports your decision
5. Keep your reasoning concise and clinical

Format your response as:
DECISION: [APPROVED or DENIED]
REASONING: [2-3 sentences explaining why]
POLICY CITATION: [specific policy requirement cited]
"""

    llama_response = call_llama(decision_prompt)
    print(f"\n{llama_response}")

    # Parse decision
    decision = "APPROVED" if "APPROVED" in llama_response.upper() else "DENIED"
    end_time = datetime.now()
    response_time_seconds = (end_time - start_time).total_seconds()

    result = {
        "patient_id": patient_id,
        "diagnosis_code": diagnosis_code,
        "procedure": procedure,
        "decision": decision,
        "reasoning": llama_response,
        "timestamp": start_time.isoformat(),
        "response_time_seconds": response_time_seconds
    }

    # ── 🏥 COMPLIANCE: Log to MLflow ──────────────────────────────
    print("\n📋 Logging decision to MLflow audit trail...")
    with mlflow.start_run(run_name=f"prior_auth_{patient_id}_{start_time.strftime('%H%M%S')}"):
        mlflow.log_param("patient_id", patient_id)
        mlflow.log_param("diagnosis_code", diagnosis_code)
        mlflow.log_param("procedure", procedure)
        mlflow.log_param("decision", decision)
        mlflow.log_metric("response_time_seconds", response_time_seconds)
        mlflow.log_text(llama_response, "agent_reasoning.txt")
        mlflow.log_text(policy_info[:2000], "policy_evidence.txt")
    print("   ✅ MLflow audit entry created!")

    # ── 🏥 COMPLIANCE: Save to Delta audit table ───────────────────
    print("📋 Saving to Delta audit log table...")
    audit_df = spark.createDataFrame(pd.DataFrame([result]))
    audit_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("prior_auth_audit_log")
    print("   ✅ Delta audit log updated!")

    print(f"\n⏱️  Total response time: {response_time_seconds:.2f} seconds")
    print(f"📊 Decision: {decision}")
    
    return result

print("✅ Prior Authorization Agent ready!")
print("✅ MLflow audit logging enabled!")
print("✅ Delta audit table logging enabled!")
print("\n🎉 Agent built successfully!")

✅ Prior Authorization Agent ready!
✅ MLflow audit logging enabled!
✅ Delta audit table logging enabled!

🎉 Agent built successfully!


In [0]:
# Cell 5 — Test the Prior Authorization Agent

# Get a real patient ID from our Gold table
sample_patient = gold_patients.iloc[0]["patient_id"]
print(f"Testing with patient: {sample_patient}")

# Test Case 1 — MRI of lumbar spine
result1 = run_prior_auth_agent(
    patient_id=sample_patient,
    diagnosis_code="M54.5",
    procedure="MRI of the lumbar spine"
)

Testing with patient: e01f1215-ad82-4281-b8ec-b7c9beac4536

PRIOR AUTHORIZATION REQUEST
Patient ID:  e01f1215-ad82-4281-b8ec-b7c9beac4536
Diagnosis:   M54.5
Procedure:   MRI of the lumbar spine

🔍 Step 1: Looking up diagnosis code...
   ICD-10 M54.5 = Low back pain

🔍 Step 2: Looking up patient history...
   Patient found in Gold table ✅

🔍 Step 3: Searching CMS policy documents...
   Relevant policy chunks retrieved ✅

🧠 Step 4: Agent reasoning and decision...



DECISION: DENIED
REASONING: The patient's low back pain diagnosis (M54.5) does not meet the CMS policy requirement for a lumbar MRI, which states that the test is only covered if the patient has not responded to a reasonable trial of conservative management lasting at least four weeks. The patient's medication list does not indicate any recent attempts at conservative management, and the procedure history does not suggest a need for a lumbar MRI. 
POLICY CITATION: LCD - Lumbar MRI (L34220) states that "MRI will be covered onl

In [0]:
# Verify audit log is permanent
count = spark.table("prior_auth_audit_log").count()
print(f"✅ Audit log has {count} permanent decision(s) stored")
print("These survive restarts and session endings")

# Show the records
spark.table("prior_auth_audit_log") \
    .select("patient_id", "diagnosis_code", "procedure", 
            "decision", "timestamp", "response_time_seconds") \
    .show(truncate=False)

✅ Audit log has 1 permanent decision(s) stored
These survive restarts and session endings
+------------------------------------+--------------+-----------------------+--------+--------------------------+---------------------+
|patient_id                          |diagnosis_code|procedure              |decision|timestamp                 |response_time_seconds|
+------------------------------------+--------------+-----------------------+--------+--------------------------+---------------------+
|e01f1215-ad82-4281-b8ec-b7c9beac4536|M54.5         |MRI of the lumbar spine|DENIED  |2026-05-20T20:05:47.014200|1.231241             |
+------------------------------------+--------------+-----------------------+--------+--------------------------+---------------------+



In [0]:
# Test Case 2 — Knee replacement
print("TEST CASE 2:")
result2 = run_prior_auth_agent(
    patient_id=gold_patients.iloc[1]["patient_id"],
    diagnosis_code="M17.1",
    procedure="Total knee replacement surgery"
)

TEST CASE 2:

PRIOR AUTHORIZATION REQUEST
Patient ID:  8dfe1ae6-88b6-46a6-bf1c-3013f7e44c0d
Diagnosis:   M17.1
Procedure:   Total knee replacement surgery

🔍 Step 1: Looking up diagnosis code...
   ICD-10 M17.1 = Primary osteoarthritis of knee

🔍 Step 2: Looking up patient history...
   Patient found in Gold table ✅

🔍 Step 3: Searching CMS policy documents...
   Relevant policy chunks retrieved ✅

🧠 Step 4: Agent reasoning and decision...



DECISION: DENIED
REASONING: The patient's primary diagnosis is primary osteoarthritis of the knee, but the requested procedure is total knee replacement surgery, which is typically considered for more severe conditions such as failure of a previous osteotomy, distal femur fracture, or malignancy. The patient's current episode of care does not indicate any of these severe conditions, and the patient's conditions and medications do not suggest a need for total knee replacement surgery.
POLICY CITATION: Covered Indications, Section 1, LCD - Major Joi

In [0]:
print("\n\nTEST CASE 3:")
# Test Case 3 — Colonoscopy
result3 = run_prior_auth_agent(
    patient_id=gold_patients.iloc[2]["patient_id"],
    diagnosis_code="Z12.11",
    procedure="Screening colonoscopy"
)



TEST CASE 3:

PRIOR AUTHORIZATION REQUEST
Patient ID:  d9ea1abc-9b19-4c5f-8917-83b3628d94e0
Diagnosis:   Z12.11
Procedure:   Screening colonoscopy

🔍 Step 1: Looking up diagnosis code...
   ICD-10 Z12.11 = Encounter for screening for colon cancer

🔍 Step 2: Looking up patient history...
   Patient found in Gold table ✅

🔍 Step 3: Searching CMS policy documents...
   Relevant policy chunks retrieved ✅

🧠 Step 4: Agent reasoning and decision...



DECISION: DENIED
REASONING: The patient's current conditions, including suspected COVID-19, acute bronchitis, and preeclampsia, do not meet the CMS coverage criteria for a screening colonoscopy, which typically requires a diagnosis of colon cancer or a related condition such as unexplained iron deficiency anemia, fecal occult blood, or clinical suspicion of inflammatory bowel disease. The patient's current conditions do not align with these criteria, and the requested procedure is not medically necessary at this time.
POLICY CITATION: LCD - D

In [0]:
# Verify all 3 decisions in audit log
print("📋 COMPLETE AUDIT LOG:")
spark.table("prior_auth_audit_log") \
    .select("patient_id", "diagnosis_code",
            "procedure", "decision",
            "response_time_seconds") \
    .show(truncate=False)

📋 COMPLETE AUDIT LOG:
+------------------------------------+--------------+------------------------------+--------+---------------------+
|patient_id                          |diagnosis_code|procedure                     |decision|response_time_seconds|
+------------------------------------+--------------+------------------------------+--------+---------------------+
|d9ea1abc-9b19-4c5f-8917-83b3628d94e0|Z12.11        |Screening colonoscopy         |DENIED  |0.885772             |
|e01f1215-ad82-4281-b8ec-b7c9beac4536|M54.5         |MRI of the lumbar spine       |DENIED  |1.231241             |
|8dfe1ae6-88b6-46a6-bf1c-3013f7e44c0d|M17.1         |Total knee replacement surgery|DENIED  |1.18838              |
+------------------------------------+--------------+------------------------------+--------+---------------------+

